# Drawing Containers

Adaptation of the original [Drawing](https://github.com/symbolfigures/drawing) repository for Docker containers and AWS services.

![diagram](diagram.drawio.png)

## 1. Source material
Drawings are scanned as PNG images and uploaded to an S3 bucket.  

Upload the scans.  
`aws s3 cp --recursive ~/symbolfigures/drawing/data/scan/secondstudy_300 \`  
`    s3://symbolfigures/drawing/scan/secondstudy_300/`

In [ ]:
# display a sample scan
from IPython.display import Image
import boto3, io
s3 = boto3.client('s3')
bucket = 'symbolfigures'
key = f'drawing/scan/secondstudy_300/00.png'
img_bytes = s3.get_object(Bucket=bucket, Key=key)['Body'].read()
display(Image(data=img_bytes))

## 2. Cut tiles
From each drawing is extracted a set of smaller drawings.
The [script](tile/main.py) downloads one scan from an S3 bucket, performs the extraction, and uploads the results to the same bucket. It runs inside a Docker container, built from an image stored in Amazon ECR. Each container is launched on ECS Fargate. An AWS Batch array job runs containers in parallel.

`cd tile`  

### a. Docker setup

Create a repository.  
`aws ecr create-repository --repository-name drawing_tile`  

Authenticate Docker to ECR  
`aws ecr get-login-password --region us-west-2 | docker login --username AWS --password-stdin $AWS_ACCOUNT_ID.dkr.ecr.us-west-2.amazonaws.com`  

Write the [dockerfile](tile/dockerfile) and build the image.  
`docker build -t drawing_tile .`  

Tag the image.  
`docker tag drawing_tile:latest $AWS_ACCOUNT_ID.dkr.ecr.us-west-2.amazonaws.com/drawing_tile:latest`  

Push the image.  
`docker push $AWS_ACCOUNT_ID.dkr.ecr.us-west-2.amazonaws.com/drawing_tile:latest`  

### b. Batch setup

Create two IAM roles from the AWS Management Console. Both have Principal Service ecs-tasks.amazonaws.com in the trust policy.
- ECSTaskExecutionRole
- ECSTaskRole-S3

Each has it's respective policy.
- [ECSTaskExecutionRolePolicy](ECSTaskExecutionRolePolicy.json)
- [ECSTaskRolePolicy-S3](ECSTaskRolePolicy-S3.json)

The first three containers run on ECS Fargate and share the same compute environment and queue.

Create the [compute environment](compute_env_fargate.json).  
`aws batch create-compute-environment \`  
`	--cli-input-json file://../compute_env_fargate.json`  

Create the job queue.  
`aws batch create-job-queue \`  
`	--job-queue-name queue_fargate \`  
`	--state ENABLED \`  
`	--priority 1 \`  
`	--compute-environment-order '[{"order": 1, "computeEnvironment": "compute_env_fargate"}]'`  

Register the [job definition](tile/job_def.json).  
`aws batch register-job-definition --cli-input-json file://job_def_drawing_tile.json`  

### c. Test

Run the python script directly and remove bugs before running in the Docker container.  
`python main.py`  

Run the Docker container locally before running on ECS. Create access keys for some IAM user in the AWS account and delete them later.  
`docker run --rm -e AWS_ACCESS_KEY_ID=$AWS_ACCESS_KEY_ID \`  
`    -e AWS_SECRET_ACCESS_KEY=$AWS_SECRET_ACCESS_KEY \`  
`    -e AWS_DEFAULT_REGION=us-west-2 \`  
`    drawing_tile:latest`  

`cd ..`  

### d. Execute

In [ ]:
import boto3
import json

batch = boto3.client('batch')

array_size = 50

response = batch.submit_job(
    jobName='batch_array_drawing_tile',
    jobQueue='queue_fargate',
    jobDefinition='job_def_drawing_tile',
    arrayProperties={'size': array_size}
)

print('submitted job:', response['jobId'])

In [ ]:
# display a sample tile
from IPython.display import Image
import boto3, io
s3 = boto3.client('s3')
bucket = 'symbolfigures'
key = 'drawing/tile/secondstudy_300/p00/t00_rf00.png'
img_bytes = s3.get_object(Bucket=bucket, Key=key)['Body'].read()
display(Image(data=img_bytes))

## 3. Rotate and flip
Rotate and flip each tile, multiplying the volume by eight.
The [script](rotateflip/main.py) downloads all the tiles for a given scan and makes eight copies: Four 90-degree rotations, and the reverse of each rotation. This is executed using the same pattern as before, and uses the same IAM roles, ECS compute environment, and ECS queue.

`cd rotateflip`

### a. Docker setup

`aws ecr create-repository --repository-name drawing_rotateflip`  
`docker build -t drawing_rotateflip .`  
`docker tag drawing_rotateflip:latest $AWS_ACCOUNT_ID.dkr.ecr.us-west-2.amazonaws.com/drawing_rotateflip:latest`  
`docker push $AWS_ACCOUNT_ID.dkr.ecr.us-west-2.amazonaws.com/drawing_rotateflip:latest`  

### b. Batch setup

`aws batch register-job-definition --cli-input-json file://job_def_drawing_rotateflip.json`  

`cd ..`  

### c. Test

### d. Execute

In [2]:
import boto3
import json

batch = boto3.client('batch')

array_size = 50

response = batch.submit_job(
    jobName='batch_array_drawing_rotateflip',
    jobQueue='queue_fargate',
    jobDefinition='job_def_drawing_rotateflip',
    arrayProperties={'size': array_size}
)

print('submitted job:', response['jobId'])

submitted job: acfa7e07-a037-4755-8cd4-8911231dfdb6


In [ ]:
# display sample rotations and flips
from IPython.display import Image
import boto3
s3 = boto3.client('s3')
bucket = 'symbolfigures'
for f in range(2):
    for r in range(4):
        key = f'drawing/tile/secondstudy_300/p00/t00_rf{f}{r}.png'
        img_bytes = s3.get_object(Bucket=bucket, Key=key)['Body'].read()
        display(Image(data=img_bytes))

## 4. Tensorflow records
Convert .png files to a .tfrecord file, to be read by the deep learning program.  

`cd tfrecord`  

### a. Docker setup

`aws ecr create-repository --repository-name drawing_tfrecord`  
`docker build -t drawing_tfrecord .`  
`docker tag drawing_tfrecord:latest $AWS_ACCOUNT_ID.dkr.ecr.us-west-2.amazonaws.com/drawing_tfrecord:latest`  
`docker push $AWS_ACCOUNT_ID.dkr.ecr.us-west-2.amazonaws.com/drawing_tfrecord:latest`  

### b. Batch setup

`aws batch register-job-definition --cli-input-json file://job_def_drawing_tfrecord.json`  

`cd ..`  

### c. Test

### d. Execute

In [3]:
import boto3
import json

batch = boto3.client('batch')

response = batch.submit_job(
    jobName='batch_array_drawing_tfrecord',
    jobQueue='queue_fargate',
    jobDefinition='job_def_drawing_tfrecord'
)

print('submitted job:', response['jobId'])

submitted job: 02636d4d-7a57-4075-bb9e-c10660459202


## 5. Train
Train the model with a generative adversarial network (GAN).

`cd train`

### a. S3 setup

Choose a name for this training batch, such as today's date, and upload the parameters file, using the name as a key partition.  
`export BATCH_NAME='2025-10-14'`  
`aws s3 cp \`  
`    ~/symbolfigures/drawing/aws/train/options.json \`  
`    s3://symbolfigures/drawing/train/secondstudy/$BATCH_NAME/options.json`

### b. Docker setup

`aws ecr create-repository --repository-name drawing_train`  
`docker build -t drawing_train .`  
`docker tag drawing_train:latest $AWS_ACCOUNT_ID.dkr.ecr.us-west-2.amazonaws.com/drawing_train:latest`  
`docker push $AWS_ACCOUNT_ID.dkr.ecr.us-west-2.amazonaws.com/drawing_train:latest`  

### c. Batch setup

This fourth container runs on EC2 and has its own compute environment and queue.

`aws batch create-compute-environment \`  
`	--cli-input-json file://../compute_env_ec2.json`  

`aws batch create-job-queue \`  
`	--job-queue-name queue_ec2 \`  
`	--state ENABLED \`  
`	--priority 1 \`  
`	--compute-environment-order '[{"order": 1, "computeEnvironment": "compute_env_ec2"}]'`  

`aws batch register-job-definition --cli-input-json file://job_def_drawing_train.json`  

`cd ..`  

### d. Test

### e. Execute

In [ ]:
import boto3
import json

batch = boto3.client('batch')

response = batch.submit_job(
    jobName='batch_drawing_train',
    jobQueue='queue_ec2',
    jobDefinition='job_def_drawing_train'
)

print('submitted job:', response['jobId'])

## 5. Generate
Generate imitation drawings with the trained model. The set of all possible outputs is a continuous vector space, so animations can be generated by performing transformations through that space. Finally, the script converts the generated sequence of images into a video, which may be downloaded from S3.

`cd gen`

### a. S3 setup

Copy the final checkpoint to the model partition and give it a name.
`aws s3 cp \`  
`    ~/symbolfigures/drawing/aws/train/512.checkpoint \`  
`    s3://symbolfigures/drawing/model/secondstudy_300_256.pkl`

### b. Docker setup

`aws ecr create-repository --repository-name drawing_gen`  
`docker build -t drawing_gen .`  
`docker tag drawing_gen:latest $AWS_ACCOUNT_ID.dkr.ecr.us-west-2.amazonaws.com/drawing_gen:latest`  
`docker push $AWS_ACCOUNT_ID.dkr.ecr.us-west-2.amazonaws.com/drawing_gen:latest`  

### c. Batch setup

`aws batch register-job-definition --cli-input-json file://job_def_drawing_gen.json`  

`cd ..`  

### d. Test

### e. Execute

In [ ]:
import boto3
import json

batch = boto3.client('batch')

response = batch.submit_job(
    jobName='batch_drawing_gen',
    jobQueue='queue_ec2',
    jobDefinition='job_def_drawing_gen'
)

print('submitted job:', response['jobId'])